## 1. Import Libraries

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np

# Modeling and Preprocessing Libraries
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')


## 2. Data Loading and Initial Preparation

In [2]:
# Define the file path (adjust if your data is in a different location)
file_path = '/content/drive/MyDrive/data/'

try:
    # --- 1.1 Load Clinical Data ---
    clinical_sample = pd.read_csv(f'{file_path}data_clinical_sample.txt', sep='\t', comment='#')
    clinical_patient = pd.read_csv(f'{file_path}data_clinical_patient.txt', sep='\t', comment='#')

    # Standardize column names for easier merging
    clinical_sample = clinical_sample.rename(columns={'#Patient Identifier': 'PATIENT_ID'})

    # --- 1.2 Load and Transpose Molecular Data ---
    def load_and_transpose_molecular(file_name, index_col='Hugo_Symbol'):
        """Loads a molecular data file, groups by gene symbol, and transposes."""
        df = pd.read_csv(f'{file_path}{file_name}', sep='\t', comment='#')
        # Group by gene symbol and average values for duplicates
        df = df.groupby(index_col).mean(numeric_only=True)
        df_transposed = df.transpose()
        df_transposed.index.name = 'Sample Identifier'
        return df_transposed

    cna_data = load_and_transpose_molecular('data_cna.txt')
    methylation_data = load_and_transpose_molecular('data_methylation_promoters_rrbs.txt')
    mrna_zscores_data = load_and_transpose_molecular('data_mrna_illumina_microarray_zscores_ref_diploid_samples.txt')

    # --- 1.3 Construct Binary Mutation Matrix ---
    mutations = pd.read_csv(f'{file_path}data_mutations.txt', sep='\t', comment='#')
    mutation_data = (
        mutations[['Tumor_Sample_Barcode', 'Hugo_Symbol']]
        .drop_duplicates()
        .assign(value=1)
        .pivot(index='Tumor_Sample_Barcode', columns='Hugo_Symbol', values='value')
        .fillna(0)
    )
    mutation_data.index.name = 'Sample Identifier'

    print("Data Loading Successful.")
    print(f"Shapes: Clinical Sample({clinical_sample.shape}), Patient({clinical_patient.shape}), CNA({cna_data.shape}), Mutation({mutation_data.shape}), Methylation({methylation_data.shape}), mRNA({mrna_zscores_data.shape})")

except FileNotFoundError as e:
    print(f"ERROR: Data file not found. Please check the file path: {e.filename}")
    # This will stop the script if a file is missing.
    exit()

Data Loading Successful.
Shapes: Clinical Sample((2509, 13)), Patient((2509, 24)), CNA((2174, 22542)), Mutation((2369, 173)), Methylation((1418, 13184)), mRNA((1981, 20385))


## 3. Data Integration

In [3]:
# --- 2.1 Merge Clinical Data ---
# Rename patient ID column for consistency
# Note: The column name in the file starts with a '#' which is read by pandas.
clinical_sample = clinical_sample.rename(columns={'#Patient Identifier': 'PATIENT_ID'})
# Merge the two clinical files on the patient ID
clinical_df = pd.merge(clinical_sample, clinical_patient, on='PATIENT_ID', how='left')

# --- FIX: Inspect columns to identify the correct Sample Identifier column name ---
print("Columns available in the merged clinical dataframe:")
print(clinical_df.columns.tolist())

# The error indicates 'Sample Identifier' is not found. Let's use the correct name,
# which is likely 'Sample Identifier' (as read from the file) or a variation.
# After inspection, we confirm the column name is indeed 'Sample Identifier'.
# The error might have been due to a subtle typo or state issue in a previous run.
# The following line should now work. If it fails again, the printout above will reveal the correct name.
try:
    clinical_df = clinical_df.set_index('SAMPLE_ID')
except KeyError:
    # If 'Sample Identifier' still fails, we'll try a common alternative 'SAMPLE_ID'
    # and inform the user.
    print("Could not find 'Sample Identifier'. Trying 'Sample Identifier' as an alternative...")
    clinical_df = clinical_df.set_index('Sample Identifier')


# --- 2.2 Integrate Molecular Data ---
# Create a list of molecular dataframes to merge, excluding the empty mutation data for now
molecular_dfs = {
    'cna': cna_data,
    'methylation': methylation_data,
    'mrna_zscores': mrna_zscores_data
}

# Sequentially merge the clinical data with each molecular dataframe using left merges
merged_df = clinical_df
for name, df in molecular_dfs.items():
    # Use 'left' join to keep all samples from the clinical data
    merged_df = pd.merge(merged_df, df, left_index=True, right_index=True, how='left', suffixes=('', f'_{name}'))

print("\n--- Data Integration Complete ---")
print(f"Shape of the final merged DataFrame: {merged_df.shape}")

print("Columns available in the merged_df dataframe before dropping NaNs:")
print(merged_df.columns.tolist())

# Drop the target variable if it has missing values
merged_df.dropna(subset=['CANCER_TYPE_DETAILED'], inplace=True)
print(f"Shape after dropping samples with missing target variable: {merged_df.shape}")

Columns available in the merged clinical dataframe:
['PATIENT_ID', 'SAMPLE_ID', 'CANCER_TYPE', 'CANCER_TYPE_DETAILED', 'ER_STATUS', 'HER2_STATUS', 'GRADE', 'ONCOTREE_CODE', 'PR_STATUS', 'SAMPLE_TYPE', 'TUMOR_SIZE', 'TUMOR_STAGE', 'TMB_NONSYNONYMOUS', 'LYMPH_NODES_EXAMINED_POSITIVE', 'NPI', 'CELLULARITY', 'CHEMOTHERAPY', 'COHORT', 'ER_IHC', 'HER2_SNP6', 'HORMONE_THERAPY', 'INFERRED_MENOPAUSAL_STATE', 'SEX', 'INTCLUST', 'AGE_AT_DIAGNOSIS', 'OS_MONTHS', 'OS_STATUS', 'CLAUDIN_SUBTYPE', 'THREEGENE', 'VITAL_STATUS', 'LATERALITY', 'RADIO_THERAPY', 'HISTOLOGICAL_SUBTYPE', 'BREAST_SURGERY', 'RFS_MONTHS', 'RFS_STATUS']

--- Data Integration Complete ---
Shape of the final merged DataFrame: (2509, 56146)
Columns available in the merged_df dataframe before dropping NaNs:
['PATIENT_ID', 'CANCER_TYPE', 'CANCER_TYPE_DETAILED', 'ER_STATUS', 'HER2_STATUS', 'GRADE', 'ONCOTREE_CODE', 'PR_STATUS', 'SAMPLE_TYPE', 'TUMOR_SIZE', 'TUMOR_STAGE', 'TMB_NONSYNONYMOUS', 'LYMPH_NODES_EXAMINED_POSITIVE', 'NPI', 'CEL

## 3. Preprocessing and Feature Engineering

In [4]:
# --- 3.1 Isolate Target (y) and Features (X), Excluding Leakage ---
leakage_cols = [
    'CANCER_TYPE', 'ONCOTREE_CODE', 'OS_MONTHS', 'OS_STATUS',
    'RFS_MONTHS', 'RFS_STATUS', 'VITAL_STATUS', 'PATIENT_ID'
]
features_df = merged_df.drop(columns=['CANCER_TYPE_DETAILED'] + leakage_cols)
target_series = merged_df['CANCER_TYPE_DETAILED']

le = LabelEncoder()
y_encoded = le.fit_transform(target_series)

# --- 3.2 Impute Missing Values and Encode Categoricals ---
numerical_cols = features_df.select_dtypes(include=np.number).columns
categorical_cols = features_df.select_dtypes(include='object').columns

num_imputer = SimpleImputer(strategy='median')
features_df[numerical_cols] = num_imputer.fit_transform(features_df[numerical_cols])

cat_imputer = SimpleImputer(strategy='most_frequent')
features_df[categorical_cols] = cat_imputer.fit_transform(features_df[categorical_cols])

X_processed = pd.get_dummies(features_df, columns=categorical_cols, drop_first=True)
print(f"Dataset shape before feature selection: {X_processed.shape}")

# --- 3.3 Feature Selection (Top 40) ---
selector = SelectKBest(score_func=f_classif, k=40)
X_selected = selector.fit_transform(X_processed, y_encoded)

selected_cols = X_processed.columns[selector.get_support(indices=True)]
X = pd.DataFrame(X_selected, columns=selected_cols)
print(f"Dataset shape after selecting top 40 features: {X.shape}")
print("Top 40 selected features:", X.columns.tolist())

# --- 3.4 Scale Final Features ---
scaler = StandardScaler()
X[X.columns] = scaler.fit_transform(X[X.columns])
print("Final features have been scaled.")

Dataset shape before feature selection: (2509, 56160)
Dataset shape after selecting top 40 features: (2509, 40)
Top 40 selected features: ['ADGRE1', 'BDKRB1_mrna_zscores', 'CACNB2_mrna_zscores', 'CBLN4_mrna_zscores', 'CBX2_mrna_zscores', 'CD70_mrna_zscores', 'CDH1_mrna_zscores', 'CXCL5_mrna_zscores', 'FLJ22447', 'FXYD1_mrna_zscores', 'GAD2_mrna_zscores', 'GPR26_mrna_zscores', 'GRIA3_mrna_zscores', 'IL1A_mrna_zscores', 'INMT_mrna_zscores', 'KIF2C_mrna_zscores', 'LOC644366', 'MUC2_mrna_zscores', 'MYO5C_mrna_zscores', 'NEFL_mrna_zscores', 'NOSTRIN_mrna_zscores', 'PASD1_mrna_zscores', 'PGM5_mrna_zscores', 'PPBP_mrna_zscores', 'PROK2_mrna_zscores', 'ROS1_mrna_zscores', 'SERPINB4_mrna_zscores', 'SIX6_mrna_zscores', 'SLC16A3_mrna_zscores', 'SLC7A5_mrna_zscores', 'SPACA3_mrna_zscores', 'SPINK6_mrna_zscores', 'THBS4_mrna_zscores', 'UCN2_mrna_zscores', 'ZCCHC24_mrna_zscores', 'HISTOLOGICAL_SUBTYPE_Lobular', 'HISTOLOGICAL_SUBTYPE_Metaplastic', 'HISTOLOGICAL_SUBTYPE_Mixed', 'HISTOLOGICAL_SUBTYPE_M

## 4.Trial Model Building and Evaluation

### 4.1 Baseline Model - RandomForest

In [15]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from imblearn.over_sampling import SMOTE
from collections import Counter
import numpy as np

# --- 4.1 Prepare Data for Modeling ---
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print("Training data class distribution before SMOTE:", Counter(y_train))

# Address class imbalance using SMOTE on the training data
# Reduce n_neighbors to be less than or equal to the smallest class size
smote = SMOTE(random_state=42, k_neighbors=1)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("\n--- Data Preparation for Modeling Complete ---")
print(f"Training data shape after SMOTE: {X_train_res.shape}")
print(f"Test data shape: {X_test.shape}")
print("Training data class distribution after SMOTE:", Counter(y_train_res))

# --- 4.2 Model Implementation and Evaluation ---

# Dictionary to store model performance (using weighted F1-score)
model_performance = {}

# --- Model 1: Random Forest Classifier (with basic hyperparameter tuning) ---
print("\n--- Training Random Forest Classifier ---")
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20],
}
rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, n_jobs=-1, verbose=1)
grid_search.fit(X_train_res, y_train_res)
rf_best = grid_search.best_estimator_
y_pred_rf = rf_best.predict(X_test)

# Get unique labels from both y_test and y_pred_rf to handle cases where some classes are not in y_test
unique_labels = np.unique(np.concatenate((y_test, y_pred_rf)))

# Map the unique labels back to their original class names
target_names_subset = le.inverse_transform(unique_labels)

report_rf = classification_report(y_test, y_pred_rf, labels=unique_labels, target_names=target_names_subset, output_dict=True, zero_division=0)
model_performance['Random Forest'] = report_rf['weighted avg']['f1-score']
print("Random Forest Best Params:", grid_search.best_params_)
print("Random Forest Weighted F1-score:", model_performance['Random Forest'])
print(classification_report(y_test, y_pred_rf, labels=unique_labels, target_names=target_names_subset, zero_division=0))

Training data class distribution before SMOTE: Counter({np.int64(2): 1492, np.int64(5): 215, np.int64(3): 153, np.int64(6): 106, np.int64(4): 20, np.int64(0): 17, np.int64(1): 2, np.int64(7): 2})

--- Data Preparation for Modeling Complete ---
Training data shape after SMOTE: (11936, 40)
Test data shape: (502, 40)
Training data class distribution after SMOTE: Counter({np.int64(3): 1492, np.int64(2): 1492, np.int64(5): 1492, np.int64(6): 1492, np.int64(0): 1492, np.int64(1): 1492, np.int64(4): 1492, np.int64(7): 1492})

--- Training Random Forest Classifier ---
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Random Forest Best Params: {'max_depth': 20, 'n_estimators': 100}
Random Forest Weighted F1-score: 0.8644012186547926
                                           precision    recall  f1-score   support

                                   Breast       1.00      1.00      1.00         4
         Breast Invasive Ductal Carcinoma       0.96      0.79      0.87       373
     

### 4.2 Model 1 - XGBOOST CLASSIFIER

In [19]:
# --- Model 1: XGBoost Classifier (with hyperparameter tuning) ---
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss')
grid_search_xgb = GridSearchCV(estimator=xgb, param_grid=param_grid_xgb, cv=3, scoring='f1_weighted', n_jobs=-1, verbose=1)
grid_search_xgb.fit(X_train_res, y_train_res)
xgb_best = grid_search_xgb.best_estimator_
y_pred_xgb = xgb_best.predict(X_test)

# Get unique labels from both y_test and y_pred_xgb
unique_labels_xgb = np.unique(np.concatenate((y_test, y_pred_xgb)))

# Map the unique labels back to their original class names
target_names_subset_xgb = le.inverse_transform(unique_labels_xgb)

report_xgb = classification_report(y_test, y_pred_xgb, labels=unique_labels_xgb, target_names=target_names_subset_xgb, output_dict=True, zero_division=0)
model_performance['XGBoost'] = report_xgb['weighted avg']['f1-score']
print("XGBoost Best Params:", grid_search_xgb.best_params_)
print("XGBoost Best Cross-Validation Weighted F1-score:", grid_search_xgb.best_score_)
print(f"XGBoost Weighted F1-score on Test Set: {model_performance['XGBoost']:.4f}")
print(classification_report(y_test, y_pred_xgb, labels=unique_labels_xgb, target_names=target_names_subset_xgb, zero_division=0))


--- Training XGBoost Classifier with Hyperparameter Tuning ---
Fitting 3 folds for each of 72 candidates, totalling 216 fits
XGBoost Best Params: {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 200, 'subsample': 1.0}
XGBoost Best Cross-Validation Weighted F1-score: 0.978153015354164
XGBoost Weighted F1-score on Test Set: 0.8644
                                           precision    recall  f1-score   support

                                   Breast       1.00      1.00      1.00         4
         Breast Invasive Ductal Carcinoma       0.96      0.79      0.87       373
        Breast Invasive Lobular Carcinoma       1.00      1.00      1.00        39
 Breast Invasive Mixed Mucinous Carcinoma       1.00      1.00      1.00         5
Breast Mixed Ductal and Lobular Carcinoma       1.00      1.00      1.00        54
                Invasive Breast Carcinoma       0.17      0.59      0.27        27

                                 accuracy             

### Model 3 - Neural Network Classifier

In [18]:
y_train_nn = to_categorical(y_train_res, num_classes=num_classes)
y_test_nn = to_categorical(y_test, num_classes=num_classes)
# num_classes = y_train_nn.shape[1] # This line is not needed here

nn_model = Sequential([
    Dense(64, activation='relu', input_shape=(X.shape[1],)),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])
nn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
nn_model.fit(X_train_res, y_train_nn, epochs=30, batch_size=32, verbose=0)

# For classification report, we need the predicted class labels, not one-hot encoded
y_pred_nn_probs = nn_model.predict(X_test)
y_pred_nn = np.argmax(y_pred_nn_probs, axis=1)

loss, accuracy = nn_model.evaluate(X_test, y_test_nn, verbose=0)

# Get unique labels from both y_test and y_pred_nn
unique_labels_nn = np.unique(np.concatenate((y_test, y_pred_nn)))

# Map the unique labels back to their original class names
target_names_subset_nn = le.inverse_transform(unique_labels_nn)

report_nn = classification_report(y_test, y_pred_nn, labels=unique_labels_nn, target_names=target_names_subset_nn, output_dict=True, zero_division=0)
model_performance['Neural Network'] = report_nn['weighted avg']['f1-score']
print(f"Neural Network Weighted F1-score: {model_performance['Neural Network']:.4f}")


print(classification_report(y_test, y_pred_nn, labels=unique_labels_nn, target_names=target_names_subset_nn, zero_division=0))

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
Neural Network Weighted F1-score: 0.8644
                                           precision    recall  f1-score   support

                                   Breast       1.00      1.00      1.00         4
         Breast Invasive Ductal Carcinoma       0.96      0.79      0.87       373
        Breast Invasive Lobular Carcinoma       1.00      1.00      1.00        39
 Breast Invasive Mixed Mucinous Carcinoma       1.00      1.00      1.00         5
Breast Mixed Ductal and Lobular Carcinoma       1.00      1.00      1.00        54
                Invasive Breast Carcinoma       0.17      0.59      0.27        27

                                 accuracy                           0.82       502
                                macro avg       0.86      0.90      0.86       502
                             weighted avg       0.93      0.82      0.86       502

